In [1]:
from model import Resnet_with_uncertainty,MLP_with_uncertainty
from second_stage_fusion_utils_concat import predict,predict3

100%|██████████| 395/395 [00:00<00:00, 1355.39it/s]


In [2]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score,accuracy_score, roc_curve,auc,average_precision_score,precision_recall_curve
import cv2
from dependency import *
import pandas as pd
from label_encoding import encode_test_labels,encode_metadata_labels

In [3]:
test_index_df = pd.read_csv(test_index_path)
train_index_df = pd.read_csv(train_index_path)
val_index_df = pd.read_csv(val_index_path)

train_index_list = list(train_index_df['indexes'])
val_index_list = list(val_index_df['indexes'])
test_index_list = list(test_index_df['indexes'])

train_index_list_1 = train_index_list[0:206]
train_index_list_2 = train_index_list[206:]

df = pd.read_csv(img_info_path)

In [4]:
import os

# Check current working directory
print("Current Directory:", os.getcwd())

# Print matching weight folders
weight_folders = [f for f in os.listdir('.') if 'weight_file' in f]
print("Found Weight Folders:", weight_folders)

Current Directory: E:\Project implementations\Project
Found Weight Folders: []


In [16]:
import random
import warnings
# Suppress deprecation warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Configurations
seed = 0
mode = 'multimodal(copy2)'
model_name = 'Resnet_cos-loss and concat'
random.seed(seed)

TTA = 6
img_type = "concat"
size = 229
data_mode = 'Normal'

# Set fold / model index
i = 2

# ---------------------------------------------------------------------
# 1. Load Image (ResNet) & Metadata (MLP) Models
# ---------------------------------------------------------------------
weight_file_resnet = './{}_{}_{}_weight_file/{}/checkpoint/best_mean_acc_model.pth'.format(mode, model_name, data_mode, i)
print(f"Loading ResNet Weights: {weight_file_resnet}")
net_resnet = Resnet_with_uncertainty(class_list).cuda()
net_resnet.load_state_dict(torch.load(weight_file_resnet))

weight_file_mlp = './{}_{}_{}_weight_file/{}/checkpoint/best_mean_acc_model.pth'.format("multimodal", "MLP", data_mode, i)
print(f"Loading MLP Weights:    {weight_file_mlp}")
net_mlp = MLP_with_uncertainty(class_list).cuda()
net_mlp.load_state_dict(torch.load(weight_file_mlp))

# Set evaluation models to test mode
net_resnet.eval()
net_mlp.eval()

# ---------------------------------------------------------------------
# 2. Run Multimodal Fusion Only (P3)
# ---------------------------------------------------------------------
print("\n========================================================")
print("  Running Multimodal Fusion (P3) Evaluation Only        ")
print("========================================================\n")

out_dir_p3 = './{}_{}_result/{}/'.format(mode, model_name, "P3")

(
    fusion_avg_acc,
    [
        fusion_diag_prob,   # Combined diagnosis probabilities
        fusion_pn_prob,     # Pigment Network probabilities
        fusion_str_prob,    # Streaks probabilities
        fusion_pig_prob,    # Pigmentation probabilities
        fusion_rs_prob,     # Regression Structures probabilities
        fusion_dag_prob,    # Dots & Globules probabilities
        fusion_bwv_prob,    # Blue Whitish Veil probabilities
        fusion_vs_prob      # Vascular Structures probabilities
    ], 
    [
        fusion_nevu_label, 
        fusion_bcc_label, 
        fusion_mel_label, 
        fusion_misc_label, 
        fusion_sk_label
    ],
    [
        fusion_nevu_prob, 
        fusion_bcc_prob, 
        fusion_mel_prob, 
        fusion_misc_prob, 
        fusion_sk_prob
    ],
    fusion_seven_point_features,
    [
        diag_gt, 
        pn_gt, 
        str_gt, 
        pig_gt, 
        rs_gt, 
        dag_gt, 
        bwv_gt, 
        vs_gt
    ]
) = predict3(
    net_resnet, 
    net_mlp, 
    test_index_list, 
    df, 
    model_name, 
    out_dir_p3, 
    mode, 
    TTA, 
    size, 
    img_type
)

# ---------------------------------------------------------------------
# 3. Quick Summary Output
# ---------------------------------------------------------------------
print("\n--- Fusion Summary Readings ---")
print(f"Multimodal Fusion Average Accuracy: {fusion_avg_acc:.4f}")
print(f"Total Samples Evaluated:            {len(diag_gt)}")

Loading ResNet Weights: ./multimodal(copy2)_Resnet_cos-loss and concat_Normal_weight_file/2/checkpoint/best_mean_acc_model.pth


e:\Project implementations\prj_env310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Project implementations\prj_env310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FileNotFoundError: [Errno 2] No such file or directory: './multimodal(copy2)_Resnet_cos-loss and concat_Normal_weight_file/2/checkpoint/best_mean_acc_model.pth'

In [14]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
seed = 0
mode = 'multimodal(copy2)'
model_name = 'Resnet_cos-loss and concat'
import random
random.seed(seed)
TTA = 6
img_type="concat"
size = 229
data_mode = 'Normal'

#change i here!
i = 2

p1_acc_list = []
p2_acc_list = []
p3_acc_list = []

mean_avg_acc_list = []

weight_file = './{}_{}_{}_weight_file/{}/checkpoint/best_mean_acc_model.pth'.format(mode,model_name,data_mode,i)
print(weight_file)

net = Resnet_with_uncertainty(class_list).cuda()
net.load_state_dict(torch.load(weight_file))

weight_file2 = './{}_{}_{}_weight_file/{}/checkpoint/best_mean_acc_model.pth'.format("multimodal","MLP",data_mode,i)
print(weight_file2)

net2 =  MLP_with_uncertainty(class_list).cuda()

net2.load_state_dict(torch.load(weight_file2))
print("P1--------------------------------on images")
out_dir = './{}_{}_result/{}/'.format(mode,model_name,"P1")
(avg_acc,uncertainty_pred,
     [prob,pn_prob,str_prob,pig_prob,rs_prob,dag_prob,bwv_prob,vs_prob], 
     [nevu_label, bcc_label, mel_label, misc_label, sk_label],
     [nevu_prob ,bcc_prob ,  mel_prob ,misc_prob ,sk_prob],
     seven_point_feature_list,
     [gt,pn_gt,str_gt,pig_gt,rs_gt,dag_gt, bwv_gt,vs_gt] )= predict(net2,net,test_index_list,df,
                                                                    model_name,out_dir,mode,TTA,size,img_type,"img")
print("P2--------------------------------on metadata")
out_dir = './{}_{}_result/{}/'.format(mode,model_name,"P2")
(avg_acc,uncertainty_pred,
     [prob,pn_prob,str_prob,pig_prob,rs_prob,dag_prob,bwv_prob,vs_prob], 
     [nevu_label, bcc_label, mel_label, misc_label, sk_label],
     [nevu_prob ,bcc_prob ,  mel_prob ,misc_prob ,sk_prob],
     seven_point_feature_list,
     [gt,pn_gt,str_gt,pig_gt,rs_gt,dag_gt, bwv_gt,vs_gt] )= predict(net2,net,test_index_list,df,
                                                                    model_name,out_dir,mode,TTA,size,img_type,"metadata")
prob_2 = prob
pn_prob_2 = pn_prob
str_prob_2 = str_prob
pig_prob_2 = pig_prob
rs_prob_2 = rs_prob
dag_prob_2 = dag_prob
bwv_prob_2 = bwv_prob
vs_prob_2 = vs_prob
uncertainty_2 = uncertainty_pred
p2_list  = [prob_2,pn_prob_2,str_prob_2,pig_prob_2,rs_prob_2,dag_prob_2,bwv_prob_2,vs_prob_2,uncertainty_2]
print("P2 Done")

print("(clinic, derm)-concat & metadata fusion here!------------------------------------------------")
out_dir = './{}_{}_result/{}/'.format(mode,model_name,"P3")
(avg_acc,
     [prob,pn_prob,str_prob,pig_prob,rs_prob,dag_prob,bwv_prob,vs_prob], 
     [nevu_label, bcc_label, mel_label, misc_label, sk_label],
     [nevu_prob ,bcc_prob ,  mel_prob ,misc_prob ,sk_prob],
     seven_point_feature_list,
     [gt,pn_gt,str_gt,pig_gt,rs_gt,dag_gt, bwv_gt,vs_gt] )= predict3(net,net2,test_index_list,df,model_name,out_dir,mode,TTA,size,img_type)


./multimodal(copy2)_Resnet_cos-loss and concat_Normal_weight_file/2/checkpoint/best_mean_acc_model.pth


e:\Project implementations\prj_env310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Project implementations\prj_env310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FileNotFoundError: [Errno 2] No such file or directory: './multimodal(copy2)_Resnet_cos-loss and concat_Normal_weight_file/2/checkpoint/best_mean_acc_model.pth'